# 🌫️ AIRE — Analisi Esplorativa dei Dati (EDA)
### Qualità dell'Aria a Milano

Questo notebook analizza i dati sulla qualità dell'aria rilevati dalle stazioni di monitoraggio di Milano nel corso del 2025.  
I dati sono stati estratti dal dataset ARPA Lombardia e normalizzati in un modello relazionale.

**Inquinanti analizzati:** C6H6, CO_8h, NO2, O3, PM10, PM25, SO2  
**Stazioni:** via Pascal, viale Liguria, viale Marche, via Senato, Verziere  
**Periodo:** Gennaio 2025 – Dicembre 2025

## 1. Importazione librerie e caricamento dati

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
import seaborn as sns
import warnings

warnings.filterwarnings('ignore')
sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams['figure.figsize'] = (12, 5)
plt.rcParams['font.size'] = 11

# Caricamento CSV
BASE = '../data/powerbi/'

misurazioni = pd.read_csv(BASE + 'misurazioni_powerbi.csv', sep=';')
stazioni    = pd.read_csv(BASE + 'stazioni_powerbi.csv', sep=';')
inquinanti  = pd.read_csv(BASE + 'inquinanti_powerbi.csv', sep=';')

# Pulizia e tipo dati
misurazioni['valore'] = misurazioni['valore'].astype(str).str.replace(',', '.').astype(float)
misurazioni['data']   = pd.to_datetime(misurazioni['data'])

# Join per avere inquinante e stazione come colonne leggibili
df = misurazioni.merge(inquinanti, left_on='inquinante_id', right_on='id_inquinante')
df = df.merge(stazioni[['id_amat', 'nome']], left_on='stazione_id', right_on='id_amat')
df.rename(columns={'nome_x': 'inquinante', 'nome_y': 'stazione'}, inplace=True)
df['mese'] = df['data'].dt.to_period('M')

print(f'Righe totali: {len(df)}')
print(f'Periodo: {df["data"].min().date()} → {df["data"].max().date()}')
print(f'Stazioni: {df["stazione"].unique()}')
print(f'Inquinanti: {df["inquinante"].unique()}')
df.head()

## 2. Statistiche descrittive

In [ ]:
stats = df.groupby('inquinante')['valore'].describe().round(2)
print("Statistiche per inquinante:")
stats

## 3. Distribuzione degli inquinanti

In [ ]:
inquinanti_list = df['inquinante'].unique()
n = len(inquinanti_list)
cols = 4
rows = (n + cols - 1) // cols

fig, axes = plt.subplots(rows, cols, figsize=(16, rows * 4))
axes = axes.flatten()

for i, inq in enumerate(inquinanti_list):
    dati = df[df['inquinante'] == inq]['valore'].dropna()
    axes[i].hist(dati, bins=30, color='steelblue', edgecolor='white', alpha=0.85)
    axes[i].set_title(f'{inq}', fontweight='bold')
    axes[i].set_xlabel('Valore')
    axes[i].set_ylabel('Frequenza')

# Nascondi assi vuoti
for j in range(i + 1, len(axes)):
    axes[j].set_visible(False)

plt.suptitle('Distribuzione dei valori per inquinante', fontsize=14, fontweight='bold', y=1.01)
plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(14, 6))
sns.boxplot(data=df, x='inquinante', y='valore', palette='Set2')
plt.title('Boxplot dei valori per inquinante', fontweight='bold')
plt.xlabel('Inquinante')
plt.ylabel('Valore (µg/m³ o mg/m³)')
plt.tight_layout()
plt.show()

## 4. Andamento temporale

In [ ]:
# Andamento mensile per tutti gli inquinanti
andamento = df.groupby(['mese', 'inquinante'])['valore'].mean().reset_index()
andamento['mese_dt'] = andamento['mese'].dt.to_timestamp()

fig, axes = plt.subplots(rows, cols, figsize=(16, rows * 4))
axes = axes.flatten()

for i, inq in enumerate(inquinanti_list):
    dati = andamento[andamento['inquinante'] == inq]
    axes[i].plot(dati['mese_dt'], dati['valore'], marker='o', linewidth=2, color='steelblue')
    axes[i].set_title(f'{inq}', fontweight='bold')
    axes[i].set_xlabel('Mese')
    axes[i].set_ylabel('Media')
    axes[i].xaxis.set_major_formatter(plt.matplotlib.dates.DateFormatter('%b'))

for j in range(i + 1, len(axes)):
    axes[j].set_visible(False)

plt.suptitle('Andamento mensile medio per inquinante — 2025', fontsize=14, fontweight='bold', y=1.01)
plt.tight_layout()
plt.show()

In [ ]:
# Confronto PM10 e NO2 sullo stesso grafico
focus = ['PM10', 'NO2']
palette = {'PM10': '#e07b54', 'NO2': '#5b8dd9'}

plt.figure(figsize=(13, 5))
for inq in focus:
    dati = andamento[andamento['inquinante'] == inq]
    plt.plot(dati['mese_dt'], dati['valore'], marker='o', label=inq,
             linewidth=2.5, color=palette[inq])

plt.title('Andamento mensile — PM10 vs NO2', fontweight='bold')
plt.xlabel('Mese')
plt.ylabel('Media µg/m³')
plt.legend()
plt.gca().xaxis.set_major_formatter(plt.matplotlib.dates.DateFormatter('%b %Y'))
plt.tight_layout()
plt.show()

## 5. Confronto tra stazioni

In [ ]:
confronto = df.groupby(['stazione', 'inquinante'])['valore'].mean().reset_index()

plt.figure(figsize=(14, 6))
sns.barplot(data=confronto, x='stazione', y='valore', hue='inquinante', palette='tab10')
plt.title('Valore medio per stazione e inquinante', fontweight='bold')
plt.xlabel('Stazione')
plt.ylabel('Media µg/m³')
plt.xticks(rotation=20, ha='right')
plt.legend(title='Inquinante', bbox_to_anchor=(1.01, 1), loc='upper left')
plt.tight_layout()
plt.show()

In [ ]:
# Heatmap stazione x inquinante
pivot = confronto.pivot(index='stazione', columns='inquinante', values='valore').round(1)

plt.figure(figsize=(11, 5))
sns.heatmap(pivot, annot=True, fmt='.1f', cmap='YlOrRd', linewidths=0.5)
plt.title('Heatmap — Valore medio per stazione e inquinante', fontweight='bold')
plt.tight_layout()
plt.show()

## 6. Superamenti dei limiti UE

In [ ]:
# Limiti giornalieri UE
limiti = {
    'PM10':  50,
    'PM25':  25,
    'NO2':  200,
    'O3':   120,
    'C6H6':   5,
    'CO_8h': 10,
    'SO2':  125
}

df['limite'] = df['inquinante'].map(limiti)
df['superamento'] = df['valore'] > df['limite']

sup_per_inq = df[df['superamento']].groupby('inquinante').size().reset_index(name='n_superamenti')
print(sup_per_inq.sort_values('n_superamenti', ascending=False))

In [ ]:
plt.figure(figsize=(10, 5))
sns.barplot(data=sup_per_inq.sort_values('n_superamenti', ascending=False),
            x='inquinante', y='n_superamenti', palette='Reds_r')
plt.title('Numero di superamenti dei limiti UE per inquinante', fontweight='bold')
plt.xlabel('Inquinante')
plt.ylabel('N° superamenti')
plt.tight_layout()
plt.show()

In [ ]:
# Superamenti per stazione
sup_stazione = df[df['superamento']].groupby('stazione').size().reset_index(name='n_superamenti')

plt.figure(figsize=(10, 5))
sns.barplot(data=sup_stazione.sort_values('n_superamenti', ascending=False),
            x='stazione', y='n_superamenti', palette='Oranges_r')
plt.title('Numero di superamenti dei limiti UE per stazione', fontweight='bold')
plt.xlabel('Stazione')
plt.ylabel('N° superamenti')
plt.xticks(rotation=20, ha='right')
plt.tight_layout()
plt.show()

In [ ]:
# Andamento mensile dei superamenti
sup_mensile = df[df['superamento']].groupby('mese').size().reset_index(name='n_superamenti')
sup_mensile['mese_dt'] = sup_mensile['mese'].dt.to_timestamp()

plt.figure(figsize=(13, 5))
plt.bar(sup_mensile['mese_dt'], sup_mensile['n_superamenti'],
        width=20, color='#e07b54', edgecolor='white')
plt.title('Superamenti mensili totali — 2025', fontweight='bold')
plt.xlabel('Mese')
plt.ylabel('N° superamenti')
plt.gca().xaxis.set_major_formatter(plt.matplotlib.dates.DateFormatter('%b %Y'))
plt.tight_layout()
plt.show()

giorni_sup = df[df['superamento']]['data'].nunique()
print(f'Giorni distinti con almeno un superamento: {giorni_sup}')
print(f'Superamenti totali: {df["superamento"].sum()}')

## 7. Conclusioni

L'analisi esplorativa dei dati sulla qualità dell'aria a Milano nel 2025 ha evidenziato i seguenti punti principali:

- **PM10 e NO2** sono gli inquinanti con la maggiore variabilità stagionale: i valori più alti si registrano nei mesi invernali (gennaio–marzo) e in autunno, mentre il periodo estivo mostra una riduzione significativa. Questo andamento è coerente con le condizioni meteorologiche tipiche della Pianura Padana, dove le inversioni termiche invernali limitano la dispersione degli inquinanti.

- **O3 (ozono)** mostra il comportamento opposto: i valori più elevati si concentrano nei mesi estivi (giugno–agosto), quando l'irraggiamento solare favorisce le reazioni fotochimiche che portano alla sua formazione.

- **Confronto tra stazioni:** alcune stazioni, in particolare quelle posizionate in zone con traffico elevato, mostrano valori medi sistematicamente più alti per NO2 e PM10 rispetto alle stazioni collocate in aree verdi o periferiche.

- **Superamenti:** il numero di giorni in cui almeno un inquinante ha superato il limite UE è concentrato nei mesi invernali. Il PM10 risulta l'inquinante con il maggior numero di superamenti assoluti durante l'anno.

- I dati confermano la necessità di monitoraggio continuo e di politiche di riduzione delle emissioni, soprattutto nel periodo ottobre–marzo.